# Notebook 06 — Stage I Capacity Sweep & Bias–Variance Diagnostics

This notebook is the next execution notebook after Notebook 04/05.

Primary goal:
- quantify how model capacity changes generalization behavior
- avoid score-chasing by explicitly tracking train/val/test gaps
- produce promotion evidence for the next notebook stage.

## Planned Tasks (Notebook 06)

- [x] Load Stage G artifacts and frozen split governance
- [x] Build shared metric helpers (AUROC, PR-AUC, Brier, ECE)
- [x] Run logistic regularization sweep (C, l1_ratio)
- [x] Run random forest capacity sweep (trees, depth, leaf size)
- [x] Create learning curves for selected candidates
- [x] Export bias–variance summary and promotion recommendation

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook06_stage_i'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook06_stage_i'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook06_stage_i'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook04_stage_g'
comparison_path = INPUT_TABLE_DIR / 'stage_g_model_stage_comparison.csv'
freeze_path = INPUT_TABLE_DIR / 'stage_g_holdout_freeze.json'

if comparison_path.exists():
    g_comparison = pd.read_csv(comparison_path)
else:
    g_comparison = pd.DataFrame()

freeze = {}
if freeze_path.exists():
    with open(freeze_path, 'r', encoding='utf-8') as f:
        freeze = json.load(f)

print('Stage I workspace ready')
print('Stage G comparison found:', comparison_path.exists())
print('Holdout freeze found:', freeze_path.exists())
g_comparison.head(5)

Stage I workspace ready
Stage G comparison found: True
Holdout freeze found: True


,stage,model,n_test,auroc,pr_auc,brier,ece_10bin
0,S0_base,elasticnet_logistic,33194,0.990228,0.972283,0.023384,0.045813
1,S0_base,random_forest,33194,0.999858,0.999278,0.003322,0.000897
2,S1_plus_clinical,elasticnet_logistic,33194,0.999889,0.999494,0.003254,0.007053
3,S1_plus_clinical,random_forest,33194,0.999998,0.999995,0.000548,0.001658
4,S2_plus_interactions_extremes,elasticnet_logistic,33194,0.999883,0.999424,0.003273,0.007083


In [2]:
# Metric helper for consistent reporting
def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        ece += abs(y_true[m].mean() - y_prob[m].mean()) * (m.sum() / len(y_true))
    return float(ece)

def metric_row(stage, model, split_name, y_true, y_prob):
    return {
        'stage': stage,
        'model': model,
        'split': split_name,
        'n': int(len(y_true)),
        'auroc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'brier': float(brier_score_loss(y_true, y_prob)),
        'ece_10bin': ece_score(y_true, y_prob)
    }

print('Metric helpers ready')

Metric helpers ready


In [3]:
# Stage I implementation block 1: canonical data assembly + frozen split + logistic sweep
np.random.seed(42)

panel_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
if not panel_path.exists():
    raise FileNotFoundError(f'Missing Stage F panel: {panel_path}')

cols = [
    'patient_id', 'day', 'I_stage_f_base', 'hazard_prob_stage_f', 'stage_f_escalation_event',
    'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active'
 ]
data = pd.read_parquet(panel_path, columns=cols).copy()

# Rebuild Stage G-compatible messy indicators deterministically for continuity
p_missing_instability = np.clip(0.03 + 0.12 * (data['response_state_active'].eq('nonresponse')).astype(float), 0.0, 0.35)
p_missing_hazard = np.clip(0.02 + 0.10 * (data['months_since_cycle_start'] > 18).astype(float), 0.0, 0.25)
p_missing_cycle = np.clip(0.01 + 0.08 * (data['stage_f_escalation_event'] == 0).astype(float), 0.0, 0.20)

u1 = np.random.rand(len(data))
u2 = np.random.rand(len(data))
u3 = np.random.rand(len(data))
data.loc[u1 < p_missing_instability, 'I_stage_f_base'] = np.nan
data.loc[u2 < p_missing_hazard, 'hazard_prob_stage_f'] = np.nan
data.loc[u3 < p_missing_cycle, 'months_since_cycle_start'] = np.nan

for c in ['I_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start']:
    data[f'{c}__is_missing'] = data[c].isna().astype(np.int8)

data['is_nonresponse'] = data['response_state_active'].eq('nonresponse').astype(np.int8)
data['is_partial'] = data['response_state_active'].eq('partial_response').astype(np.int8)
data['is_stabilized'] = data['response_state_active'].eq('stabilized').astype(np.int8)

data['I_stage_f_base_x_cycle'] = data['I_stage_f_base'].fillna(data['I_stage_f_base'].median()) * data['cycle_id_stage_f']
data['extreme_hazard_flag'] = (
    data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median())
    >= data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median()).quantile(0.995)
).astype(np.int8)

label = 'stage_f_escalation_event'

# keep manageable training sample with class balance
pos = data[data[label] == 1]
neg = data[data[label] == 0].sample(n=min(len(data[data[label] == 0]), len(pos) * 3), random_state=42)
model_df = pd.concat([pos, neg], axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)

# frozen patient-wise split reconstruction
patients = model_df['patient_id'].drop_duplicates().sample(frac=1.0, random_state=42).to_numpy()
n = len(patients)
p_train = set(patients[: int(0.70 * n)])
p_val = set(patients[int(0.70 * n): int(0.85 * n)])
p_test = set(patients[int(0.85 * n):])

model_df['split'] = np.where(
    model_df['patient_id'].isin(p_train),
    'train',
    np.where(model_df['patient_id'].isin(p_val), 'val', 'test')
)

feature_set = [
    'cycle_id_stage_f', 'months_since_cycle_start', 'I_stage_f_base', 'hazard_prob_stage_f',
    'is_nonresponse', 'is_partial', 'is_stabilized',
    'I_stage_f_base__is_missing', 'hazard_prob_stage_f__is_missing', 'months_since_cycle_start__is_missing',
    'I_stage_f_base_x_cycle', 'extreme_hazard_flag'
 ]

X_train = model_df.loc[model_df['split'] == 'train', feature_set]
y_train = model_df.loc[model_df['split'] == 'train', label].astype(int).to_numpy()
X_val = model_df.loc[model_df['split'] == 'val', feature_set]
y_val = model_df.loc[model_df['split'] == 'val', label].astype(int).to_numpy()
X_test = model_df.loc[model_df['split'] == 'test', feature_set]
y_test = model_df.loc[model_df['split'] == 'test', label].astype(int).to_numpy()

pre = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), feature_set)
], remainder='drop')

logistic_grid = [
    {'C': 0.2, 'l1_ratio': 0.10},
    {'C': 0.4, 'l1_ratio': 0.25},
    {'C': 0.8, 'l1_ratio': 0.35},
    {'C': 1.5, 'l1_ratio': 0.50},
    {'C': 3.0, 'l1_ratio': 0.70}
 ]

rows = []
for cfg in logistic_grid:
    model = Pipeline([
        ('pre', pre),
        ('clf', LogisticRegression(
            l1_ratio=cfg['l1_ratio'],
            C=cfg['C'],
            solver='saga',
            max_iter=800
        ))
    ])
    model.fit(X_train, y_train)

    p_tr = model.predict_proba(X_train)[:, 1]
    p_va = model.predict_proba(X_val)[:, 1]
    p_te = model.predict_proba(X_test)[:, 1]

    rows.append(metric_row('I_logistic_sweep', f"elasticnet_C{cfg['C']}_l1r{cfg['l1_ratio']}", 'train', y_train, p_tr))
    rows.append(metric_row('I_logistic_sweep', f"elasticnet_C{cfg['C']}_l1r{cfg['l1_ratio']}", 'val', y_val, p_va))
    rows.append(metric_row('I_logistic_sweep', f"elasticnet_C{cfg['C']}_l1r{cfg['l1_ratio']}", 'test', y_test, p_te))

sweep = pd.DataFrame(rows)
sweep.to_csv(TABLE_DIR / 'stage_i_logistic_capacity_sweep.csv', index=False)

# simple overfit gap summary on AUROC
gap_rows = []
for name, g in sweep.groupby('model'):
    tr = float(g.loc[g['split'] == 'train', 'auroc'].iloc[0])
    va = float(g.loc[g['split'] == 'val', 'auroc'].iloc[0])
    te = float(g.loc[g['split'] == 'test', 'auroc'].iloc[0])
    gap_rows.append({
        'model': name,
        'train_auroc': tr,
        'val_auroc': va,
        'test_auroc': te,
        'train_val_gap': tr - va,
        'val_test_gap': va - te
    })

gap_df = pd.DataFrame(gap_rows).sort_values('val_auroc', ascending=False)
gap_df.to_csv(TABLE_DIR / 'stage_i_logistic_gap_summary.csv', index=False)

best = gap_df.iloc[0].to_dict()
with open(REPORT_DIR / 'stage_i_logistic_sweep_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage I Logistic Capacity Sweep Summary\n')
    f.write(f"rows_model_df: {len(model_df)}\n")
    f.write(f"n_train: {(model_df['split']=='train').sum()}\n")
    f.write(f"n_val: {(model_df['split']=='val').sum()}\n")
    f.write(f"n_test: {(model_df['split']=='test').sum()}\n")
    f.write(f"best_model_by_val_auroc: {best['model']}\n")
    f.write(f"best_val_auroc: {best['val_auroc']:.6f}\n")
    f.write(f"best_test_auroc: {best['test_auroc']:.6f}\n")
    f.write(f"best_train_val_gap: {best['train_val_gap']:.6f}\n")

plt.figure(figsize=(9,5))
plot_df = gap_df.head(10).copy()
plt.plot(plot_df['model'], plot_df['train_auroc'], marker='o', label='train')
plt.plot(plot_df['model'], plot_df['val_auroc'], marker='o', label='val')
plt.plot(plot_df['model'], plot_df['test_auroc'], marker='o', label='test')
plt.xticks(rotation=35, ha='right')
plt.ylabel('AUROC')
plt.title('Stage I Logistic Capacity Sweep (Train/Val/Test)')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_i_logistic_capacity_sweep.png', dpi=140, bbox_inches='tight')
plt.close()

print('Stage I block 1 generated:')
print('-', (TABLE_DIR / 'stage_i_logistic_capacity_sweep.csv').exists())
print('-', (TABLE_DIR / 'stage_i_logistic_gap_summary.csv').exists())
print('-', (REPORT_DIR / 'stage_i_logistic_sweep_summary.txt').exists())
print('-', (FIG_DIR / 'stage_i_logistic_capacity_sweep.png').exists())

gap_df.head(10)

c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Hope\miniforge3\envs\ml-ultra\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Stage I block 1 generated:
- True
- True
- True
- True


,model,train_auroc,val_auroc,test_auroc,train_val_gap,val_test_gap
4,elasticnet_C3.0_l1r0.7,0.999990,0.999993,0.999993,-0.000003,9.422047e-07
3,elasticnet_C1.5_l1r0.5,0.999983,0.999984,0.999985,-0.000002,-2.663741e-07
2,elasticnet_C0.8_l1r0.35,0.999923,0.999937,0.999908,-0.000015,2.971357e-05
1,elasticnet_C0.4_l1r0.25,0.999753,0.999814,0.999784,-0.000061,2.920323e-05
0,elasticnet_C0.2_l1r0.1,0.999510,0.999569,0.999516,-0.000059,5.236473e-05


In [4]:
# Stage I implementation block 2: random forest capacity sweep
rf_grid = [
    {'n_estimators': 120, 'max_depth': 6, 'min_samples_leaf': 20},
    {'n_estimators': 180, 'max_depth': 8, 'min_samples_leaf': 12},
    {'n_estimators': 240, 'max_depth': 10, 'min_samples_leaf': 8},
    {'n_estimators': 320, 'max_depth': 12, 'min_samples_leaf': 5},
    {'n_estimators': 420, 'max_depth': None, 'min_samples_leaf': 3}
 ]

rf_rows = []
for cfg in rf_grid:
    rf_pre = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), feature_set)
    ], remainder='drop')

    rf_model = Pipeline([
        ('pre', rf_pre),
        ('clf', RandomForestClassifier(
            n_estimators=cfg['n_estimators'],
            max_depth=cfg['max_depth'],
            min_samples_leaf=cfg['min_samples_leaf'],
            random_state=42,
            n_jobs=-1
        ))
    ])
    rf_model.fit(X_train, y_train)

    p_tr = rf_model.predict_proba(X_train)[:, 1]
    p_va = rf_model.predict_proba(X_val)[:, 1]
    p_te = rf_model.predict_proba(X_test)[:, 1]

    name = f"rf_n{cfg['n_estimators']}_d{cfg['max_depth']}_leaf{cfg['min_samples_leaf']}"
    rf_rows.append(metric_row('I_rf_sweep', name, 'train', y_train, p_tr))
    rf_rows.append(metric_row('I_rf_sweep', name, 'val', y_val, p_va))
    rf_rows.append(metric_row('I_rf_sweep', name, 'test', y_test, p_te))

rf_sweep = pd.DataFrame(rf_rows)
rf_sweep.to_csv(TABLE_DIR / 'stage_i_rf_capacity_sweep.csv', index=False)

rf_gap_rows = []
for name, g in rf_sweep.groupby('model'):
    tr = float(g.loc[g['split'] == 'train', 'auroc'].iloc[0])
    va = float(g.loc[g['split'] == 'val', 'auroc'].iloc[0])
    te = float(g.loc[g['split'] == 'test', 'auroc'].iloc[0])
    rf_gap_rows.append({
        'model': name,
        'train_auroc': tr,
        'val_auroc': va,
        'test_auroc': te,
        'train_val_gap': tr - va,
        'val_test_gap': va - te
    })

rf_gap_df = pd.DataFrame(rf_gap_rows).sort_values('val_auroc', ascending=False)
rf_gap_df.to_csv(TABLE_DIR / 'stage_i_rf_gap_summary.csv', index=False)

rf_best = rf_gap_df.iloc[0].to_dict()
with open(REPORT_DIR / 'stage_i_rf_sweep_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage I RandomForest Capacity Sweep Summary\n')
    f.write(f"best_model_by_val_auroc: {rf_best['model']}\n")
    f.write(f"best_val_auroc: {rf_best['val_auroc']:.6f}\n")
    f.write(f"best_test_auroc: {rf_best['test_auroc']:.6f}\n")
    f.write(f"best_train_val_gap: {rf_best['train_val_gap']:.6f}\n")

plt.figure(figsize=(9,5))
plot_df = rf_gap_df.head(10).copy()
plt.plot(plot_df['model'], plot_df['train_auroc'], marker='o', label='train')
plt.plot(plot_df['model'], plot_df['val_auroc'], marker='o', label='val')
plt.plot(plot_df['model'], plot_df['test_auroc'], marker='o', label='test')
plt.xticks(rotation=35, ha='right')
plt.ylabel('AUROC')
plt.title('Stage I RF Capacity Sweep (Train/Val/Test)')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_i_rf_capacity_sweep.png', dpi=140, bbox_inches='tight')
plt.close()

print('Stage I block 2 generated:')
print('-', (TABLE_DIR / 'stage_i_rf_capacity_sweep.csv').exists())
print('-', (TABLE_DIR / 'stage_i_rf_gap_summary.csv').exists())
print('-', (REPORT_DIR / 'stage_i_rf_sweep_summary.txt').exists())
print('-', (FIG_DIR / 'stage_i_rf_capacity_sweep.png').exists())

rf_gap_df.head(10)

Stage I block 2 generated:
- True
- True
- True
- True


,model,train_auroc,val_auroc,test_auroc,train_val_gap,val_test_gap
3,rf_n320_d12_leaf5,1.000000,0.999999,0.999999,6.519319e-07,-1.954782e-08
4,rf_n420_dNone_leaf3,1.000000,0.999999,0.999999,9.345103e-07,-3.649145e-07
2,rf_n240_d10_leaf8,0.999999,0.999998,0.999999,8.461721e-07,-3.760051e-07
1,rf_n180_d8_leaf12,0.999998,0.999997,0.999997,1.254911e-06,-4.583131e-07
0,rf_n120_d6_leaf20,0.999992,0.999993,0.999993,-7.396204e-07,-4.678012e-07


In [6]:
# Stage I consolidation: combined recommendation + manifest/proof
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
META_DIR.mkdir(parents=True, exist_ok=True)

log_gap = pd.read_csv(TABLE_DIR / 'stage_i_logistic_gap_summary.csv')
rf_gap = pd.read_csv(TABLE_DIR / 'stage_i_rf_gap_summary.csv')

log_best = log_gap.sort_values('val_auroc', ascending=False).iloc[0]
rf_best = rf_gap.sort_values('val_auroc', ascending=False).iloc[0]

candidates = pd.DataFrame([
    {'family': 'elasticnet', **log_best.to_dict()},
    {'family': 'random_forest', **rf_best.to_dict()}
])
candidates['stability_score'] = 1.0 - candidates['train_val_gap'].abs() - candidates['val_test_gap'].abs()
chosen = candidates.sort_values(['val_auroc', 'stability_score'], ascending=[False, False]).iloc[0]

summary_lines = [
    'Stage I Capacity & Bias-Variance Consolidation',
    f"best_elasticnet: {log_best['model']}",
    f"best_elasticnet_val_auroc: {log_best['val_auroc']:.6f}",
    f"best_rf: {rf_best['model']}",
    f"best_rf_val_auroc: {rf_best['val_auroc']:.6f}",
    f"selected_family_for_next_stage: {chosen['family']}",
    f"selected_model: {chosen['model']}",
    f"selected_val_auroc: {chosen['val_auroc']:.6f}",
    f"selected_test_auroc: {chosen['test_auroc']:.6f}",
    f"selected_train_val_gap: {chosen['train_val_gap']:.6e}",
    f"selected_val_test_gap: {chosen['val_test_gap']:.6e}"
]
with open(REPORT_DIR / 'stage_i_capacity_consolidation.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))

manifest_i = {
    'phase': 'I',
    'notebook': '06_stage_i_capacity_bias_variance.ipynb',
    'inputs': [
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet',
        'Results/tables/notebook04_stage_g/stage_g_holdout_freeze.json'
    ],
    'outputs_tables': [
        'Results/tables/notebook06_stage_i/stage_i_logistic_capacity_sweep.csv',
        'Results/tables/notebook06_stage_i/stage_i_logistic_gap_summary.csv',
        'Results/tables/notebook06_stage_i/stage_i_rf_capacity_sweep.csv',
        'Results/tables/notebook06_stage_i/stage_i_rf_gap_summary.csv'
    ],
    'outputs_figures': [
        'Results/figures/notebook06_stage_i/stage_i_logistic_capacity_sweep.png',
        'Results/figures/notebook06_stage_i/stage_i_rf_capacity_sweep.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook06_stage_i/stage_i_logistic_sweep_summary.txt',
        'Results/reports/notebook06_stage_i/stage_i_rf_sweep_summary.txt',
        'Results/reports/notebook06_stage_i/stage_i_capacity_consolidation.txt'
    ]
}
with open(META_DIR / 'phase_i_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_i, f, indent=4)

proof_i = {
    'logistic_sweep_generated': (TABLE_DIR / 'stage_i_logistic_capacity_sweep.csv').exists(),
    'rf_sweep_generated': (TABLE_DIR / 'stage_i_rf_capacity_sweep.csv').exists(),
    'consolidation_report_generated': (REPORT_DIR / 'stage_i_capacity_consolidation.txt').exists(),
    'manifest_generated': (META_DIR / 'phase_i_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_i_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_i, 'selected': chosen.to_dict()}, f, indent=4)

print('Stage I consolidation generated')
for k, v in proof_i.items():
    print('-', k, ':', v)

candidates.sort_values(['val_auroc', 'stability_score'], ascending=[False, False])

Stage I consolidation generated
- logistic_sweep_generated : True
- rf_sweep_generated : True
- consolidation_report_generated : True
- manifest_generated : True


,family,model,train_auroc,val_auroc,test_auroc,train_val_gap,val_test_gap,stability_score
1,random_forest,rf_n320_d12_leaf5,1.00000,0.999999,0.999999,6.519319e-07,-1.954782e-08,0.999999
0,elasticnet,elasticnet_C3.0_l1r0.7,0.99999,0.999993,0.999993,-3.448534e-06,9.422047e-07,0.999996


In [7]:
# Stage I implementation block 3: learning curves + manifest/proof augmentation
import re
from sklearn.model_selection import GroupKFold, learning_curve

required_vars = ['model_df', 'feature_set', 'label', 'TABLE_DIR', 'FIG_DIR', 'REPORT_DIR', 'META_DIR']
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required variables {missing}. Run Cells 3–7 first to rebuild Stage I context."
    )

log_gap = pd.read_csv(TABLE_DIR / 'stage_i_logistic_gap_summary.csv').sort_values('val_auroc', ascending=False)
rf_gap = pd.read_csv(TABLE_DIR / 'stage_i_rf_gap_summary.csv').sort_values('val_auroc', ascending=False)
best_log_name = str(log_gap.iloc[0]['model'])
best_rf_name = str(rf_gap.iloc[0]['model'])

def build_model_from_name(model_name, features):
    if model_name.startswith('elasticnet_C'):
        m = re.match(r'elasticnet_C([0-9\.]+)_l1r([0-9\.]+)', model_name)
        if not m:
            raise ValueError(f'Cannot parse elasticnet model name: {model_name}')
        C = float(m.group(1))
        l1r = float(m.group(2))
        pre = ColumnTransformer([
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), features)
        ], remainder='drop')
        return Pipeline([
            ('pre', pre),
            ('clf', LogisticRegression(
                l1_ratio=l1r,
                C=C,
                solver='saga',
                max_iter=1200
            ))
        ])

    if model_name.startswith('rf_n'):
        m = re.match(r'rf_n(\d+)_d([^_]+)_leaf(\d+)', model_name)
        if not m:
            raise ValueError(f'Cannot parse RF model name: {model_name}')
        n_estimators = int(m.group(1))
        depth_raw = m.group(2)
        max_depth = None if depth_raw == 'None' else int(depth_raw)
        min_samples_leaf = int(m.group(3))

        rf_pre = ColumnTransformer([
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), features)
        ], remainder='drop')

        return Pipeline([
            ('pre', rf_pre),
            ('clf', RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                random_state=42,
                n_jobs=-1
            ))
        ])

    raise ValueError(f'Unsupported model naming format: {model_name}')

curve_candidates = [
    ('elasticnet', best_log_name),
    ('random_forest', best_rf_name),
]

X_all = model_df[feature_set]
y_all = model_df[label].astype(int).to_numpy()
groups = model_df['patient_id'].to_numpy()

unique_groups = pd.Series(groups).nunique()
n_splits = 5 if unique_groups >= 5 else max(2, int(unique_groups))
cv = GroupKFold(n_splits=n_splits)
train_sizes = np.linspace(0.2, 1.0, 5)

curve_rows = []
for family, model_name in curve_candidates:
    estimator = build_model_from_name(model_name, feature_set)
    sizes, train_scores, val_scores = learning_curve(
        estimator=estimator,
        X=X_all,
        y=y_all,
        groups=groups,
        cv=cv,
        train_sizes=train_sizes,
        scoring='roc_auc',
        n_jobs=-1,
        shuffle=True,
        random_state=42
    )

    for idx, s in enumerate(sizes):
        curve_rows.append({
            'family': family,
            'model': model_name,
            'train_size': int(s),
            'train_auroc_mean': float(np.mean(train_scores[idx])),
            'train_auroc_std': float(np.std(train_scores[idx])),
            'val_auroc_mean': float(np.mean(val_scores[idx])),
            'val_auroc_std': float(np.std(val_scores[idx])),
            'generalization_gap': float(np.mean(train_scores[idx]) - np.mean(val_scores[idx]))
        })

curve_df = pd.DataFrame(curve_rows)
curve_df.to_csv(TABLE_DIR / 'stage_i_learning_curves.csv', index=False)

plt.figure(figsize=(9, 5))
for model_name, g in curve_df.groupby('model'):
    g = g.sort_values('train_size')
    plt.plot(g['train_size'], g['train_auroc_mean'], marker='o', label=f"{model_name} train")
    plt.plot(g['train_size'], g['val_auroc_mean'], marker='o', linestyle='--', label=f"{model_name} val")
plt.xlabel('Train size')
plt.ylabel('AUROC')
plt.title('Stage I Learning Curves (Top ElasticNet vs Top RF)')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_i_learning_curves.png', dpi=140, bbox_inches='tight')
plt.close()

final_rows = []
for model_name, g in curve_df.groupby('model'):
    g = g.sort_values('train_size')
    r = g.iloc[-1]
    final_rows.append({
        'model': model_name,
        'final_train_size': int(r['train_size']),
        'final_train_auroc': float(r['train_auroc_mean']),
        'final_val_auroc': float(r['val_auroc_mean']),
        'final_gap': float(r['generalization_gap'])
    })
final_df = pd.DataFrame(final_rows).sort_values('final_val_auroc', ascending=False)

with open(REPORT_DIR / 'stage_i_learning_curve_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage I Learning Curve Summary\n')
    f.write(f'n_unique_patients: {int(unique_groups)}\n')
    f.write(f'cv_splits: {int(n_splits)}\n')
    for _, row in final_df.iterrows():
        f.write(f"model: {row['model']} | final_train_auroc: {row['final_train_auroc']:.6f} | final_val_auroc: {row['final_val_auroc']:.6f} | final_gap: {row['final_gap']:.6f}\n")

manifest_path = META_DIR / 'phase_i_manifest.json'
if manifest_path.exists():
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest_i = json.load(f)
else:
    manifest_i = {'phase': 'I', 'notebook': '06_stage_i_capacity_bias_variance.ipynb'}

manifest_i.setdefault('outputs_tables', [])
manifest_i.setdefault('outputs_figures', [])
manifest_i.setdefault('outputs_reports', [])

learning_table = 'Results/tables/notebook06_stage_i/stage_i_learning_curves.csv'
learning_figure = 'Results/figures/notebook06_stage_i/stage_i_learning_curves.png'
learning_report = 'Results/reports/notebook06_stage_i/stage_i_learning_curve_summary.txt'

if learning_table not in manifest_i['outputs_tables']:
    manifest_i['outputs_tables'].append(learning_table)
if learning_figure not in manifest_i['outputs_figures']:
    manifest_i['outputs_figures'].append(learning_figure)
if learning_report not in manifest_i['outputs_reports']:
    manifest_i['outputs_reports'].append(learning_report)

with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest_i, f, indent=4)

proof_path = REPORT_DIR / 'stage_i_checklist_proof.json'
if proof_path.exists():
    with open(proof_path, 'r', encoding='utf-8') as f:
        proof_payload = json.load(f)
else:
    proof_payload = {'proof': {}, 'selected': {}}

proof_payload.setdefault('proof', {})
proof_payload['proof']['learning_curves_generated'] = (TABLE_DIR / 'stage_i_learning_curves.csv').exists()
proof_payload['proof']['learning_curve_plot_generated'] = (FIG_DIR / 'stage_i_learning_curves.png').exists()
proof_payload['proof']['learning_curve_report_generated'] = (REPORT_DIR / 'stage_i_learning_curve_summary.txt').exists()

with open(proof_path, 'w', encoding='utf-8') as f:
    json.dump(proof_payload, f, indent=4)

print('Stage I block 3 generated:')
print('-', (TABLE_DIR / 'stage_i_learning_curves.csv').exists())
print('-', (FIG_DIR / 'stage_i_learning_curves.png').exists())
print('-', (REPORT_DIR / 'stage_i_learning_curve_summary.txt').exists())
print('-', manifest_path.exists())
print('-', proof_path.exists())

curve_df.head(10)

Stage I block 3 generated:
- True
- True
- True
- True
- True


,family,model,train_size,train_auroc_mean,train_auroc_std,val_auroc_mean,val_auroc_std,generalization_gap
0,elasticnet,elasticnet_C3.0_l1r0.7,35290,0.999939,1.324546e-05,0.999925,2.281931e-05,1.384763e-05
1,elasticnet,elasticnet_C3.0_l1r0.7,70581,0.999983,2.187874e-06,0.999983,4.880926e-06,4.189596e-07
2,elasticnet,elasticnet_C3.0_l1r0.7,105872,0.999989,9.849743e-07,0.999989,3.285373e-06,-9.351905e-09
3,elasticnet,elasticnet_C3.0_l1r0.7,141163,0.999991,4.577415e-07,0.999991,2.643804e-06,1.184833e-08
4,elasticnet,elasticnet_C3.0_l1r0.7,176454,0.999993,3.359723e-07,0.999992,2.012267e-06,3.492590e-07
5,random_forest,rf_n320_d12_leaf5,35290,0.999999,1.220415e-07,0.999996,2.411266e-06,3.558687e-06
6,random_forest,rf_n320_d12_leaf5,70581,1.000000,1.601783e-07,0.999997,1.282213e-06,2.252030e-06
7,random_forest,rf_n320_d12_leaf5,105872,1.000000,7.461148e-08,0.999998,9.421646e-07,1.497473e-06
8,random_forest,rf_n320_d12_leaf5,141163,1.000000,7.300058e-08,0.999998,6.857430e-07,1.118101e-06
9,random_forest,rf_n320_d12_leaf5,176454,1.000000,4.350343e-08,0.999999,7.479474e-07,1.046538e-06


## Implementation Notes

Coding starts here in the next step:
1. Bring in canonical contract data via adapter output
2. Rebuild the frozen train/val/test partitions
3. Execute parameter sweeps and learning curves
4. Save `stage_i_capacity_sweep.csv` + `stage_i_learning_curves.csv` + `stage_i_bias_variance_summary.txt`

In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook06'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)